<a href="https://colab.research.google.com/github/nat-petty/MMA3001-project/blob/main/MMA3001projectNP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**MMA3001 — Combined Cycle Power Plant Output Prediction**

Aim: Predict net electrical output (PE, MW) using Average Temperature (AT) (°C), Exhaust Vacuum (V) (cm Hg), Ambient Pressure (AP) (mbar), Relative Humidity (RH) (%). Compare a linear baseline with nonlinear models and validate using the five provided shuffled sheets.

Note: Five sheets are reorderings of the same 9,568 records, not 47,840 independent samples. Never concatenate them to increase the dataset size. Use Sheet1 for exploratory analysis and model selection; use all five sheets for the supplied 5×2 validation protocol. This is not chronological forecasting: no timestamps are provided. Results apply to full-load operation within the sampled conditions.



In [ ]:
%pip -q install pandas odfpy scikit-learn matplotlib seaborn joblib pytest pdoc

from pathlib import Path
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from joblib import dump
SEED = 42
sns.set_theme(style='whitegrid')
DATA = Path('data/Folds5x2_pp.ods')
if not DATA.exists():
    DATA = Path('Folds5x2_pp.ods')
if not DATA.exists():
    try:
        from google.colab import files
        print('Upload Folds5x2_pp.ods')
        uploaded = files.upload()
        DATA = Path('Folds5x2_pp.ods')
    except ImportError:
        raise FileNotFoundError('Place Folds5x2_pp.ods in data/ or current directory')
assert DATA.exists(), 'Dataset not found'
OUTPUT = Path('results'); OUTPUT.mkdir(exist_ok=True)
sheets = pd.read_excel(DATA, sheet_name=None, engine='odf')
print('Sheets:', {name: df.shape for name, df in sheets.items()})
df = sheets['Sheet1'].copy()
display(df.head())

**Verify dataset integrity and data dictionary**

Check dimensions, data types, missing/non-finite values, duplicates, and whether all five sheets contain the same observations. Duplicates are reported rather than automatically deleted because repeated operating states can be genuine. No imputing is done without evidence.

In [ ]:
FEATURES = ['AT','V','AP','RH']; TARGET = 'PE'
expected = FEATURES + [TARGET]
for name, frame in sheets.items():
    assert list(frame.columns) == expected, (name, frame.columns.tolist())
    assert frame.shape == (9568,5), (name,frame.shape)
    assert all(pd.api.types.is_numeric_dtype(frame[c]) for c in expected)
    assert np.isfinite(frame[expected].to_numpy()).all(), f'Nonfinite values in {name}'
reference = df.value_counts(dropna=False).sort_index()
for name, frame in sheets.items():
    assert reference.equals(frame.value_counts(dropna=False).sort_index()), f'{name} is not the same multiset'
print('Missing values:\n', df.isna().sum())
print('Exact duplicate rows:',df.duplicated().sum())
display(df.describe().T)
display(df.corr(numeric_only=True).round(3))